In [2]:
import os

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

# 大模型
llm = init_chat_model(
    "deepseek-chat",
    temperature=0.1,
    api_key=os.environ.get("DEEPSEEK_API"),
    base_url="https://api.deepseek.com/v1",
    model_provider="openai",
)


# 工具函数
@tool
def calulate(expression: str) -> str:
    """计算一个数学表达式的结果"""
    try:
        result = str(eval(expression))
    except Exception as e:
        result = f"计算错误: {e}"


@tool
def get_info(city_name: str) -> str:
    """获取城市信息

    Args:
        city_name (str): _description_

    Returns:
        str: _description_
    """
    city_data = {
        "北京": "北京是中国的首都，拥有丰富的历史和文化遗产。",
        "上海": "上海是中国最大的城市之一，以其现代化的城市景观和繁华的商业区而闻名。",
        "广州": "广州是中国南方的重要城市，以其美食和商业活动而著名。",
        "深圳": "深圳是中国改革开放的前沿城市，以其创新和高科技产业而闻名。",
        "杭州": "杭州是中国东部的一个美丽城市，以西湖和茶文化而闻名。",
    }
    return city_data.get(city_name, f"未找到关于{city_name}的信息。")


# 创建一个智能体
agent = create_agent(
    model=llm,
    tools=[calulate, get_info],
    system_prompt="你是一个智能助手，可以回答用户的问题，并使用工具函数来计算数学表达式或获取城市信息。",
)

from rich.console import Console
from rich.panel import Panel

console = Console()

# 测试智能体
user_input = "请计算 3 + 5 的结果"
response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
console.print(
    Panel(
        f"[bold green]用户输入:[/bold green] {user_input}\n[bold yellow]最终回复:[/bold yellow] {response['messages'][-1].content}\n[bold orange]智能体回复:[/bold orange] {response}"
    )
)

console.print(
    Panel(
        title="消息列表",
        renderable="\n".join([
            f"[bold green]消息 {i}:[/bold green] {type(msg).__name__}: {msg}"
            for i, msg in enumerate(response["messages"])
        ]),
    )
)

# 查询城市信息

user_input = "请告诉我关于北京的信息"
response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
console.print(
    Panel(
        f"[bold green]用户输入:[/bold green] {user_input}\n[bold yellow]最终回复:[/bold yellow] {response['messages'][-1].content}\n[bold orange]智能体回复:[/bold orange] {response}"
    )
)
console.print(
    Panel(
        title="消息列表",
        renderable="\n".join([
            f"[bold green]消息 {i}:[/bold green] {type(msg).__name__}: {msg}"
            for i, msg in enumerate(response["messages"])
        ]),
    )
)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 用户输入: 请计算 3 + 5 的结果                                                                                   │
│ 最终回复: 3 + 5 的结果是 **8**。                                                                                │
│ 智能体回复: {'messages': [HumanMessage(content='请计算 3 + 5 的结果', additional_kwargs={},                     │
│ response_metadata={}, id='240739c4-ce09-4c4e-9037-398f6b17515b'), AIMessage(content='好的，我来计算 3 + 5       │
│ 的结果。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57,     │
│ 'prompt_tokens': 367, 'total_tokens': 424, 'completion_tokens_details': None, 'prompt_tokens_details':          │
│ {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 111}, │
│ 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint':                            │
│ 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '0db8af1a-9b2a-4055-9d8c-4e6db35b5c78', 'finish_reason':   │
│ 'tool_calls', 'logprobs': None}, id='lc_run--019f2157-178e-7283-a3b4-d8f2fad0a3e4-0', tool_calls=[{'name':      │
│ 'calulate', 'args': {'expression': '3 + 5'}, 'id': 'call_00_NOw4HLaJi1SZrTSUdHhp5217', 'type': 'tool_call'}],   │
│ invalid_tool_calls=[], usage_metadata={'input_tokens': 367, 'output_tokens': 57, 'total_tokens': 424,           │
│ 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}), ToolMessage(content='null',           │
│ name='calulate', id='f24ed59f-2aa1-4f9a-9407-4f9caff53f41', tool_call_id='call_00_NOw4HLaJi1SZrTSUdHhp5217'),   │
│ AIMessage(content='3 + 5 的结果是 **8**。', additional_kwargs={'refusal': None},                                │
│ response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 437, 'total_tokens': 448,          │
│ 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},       │
│ 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 53}, 'model_provider': 'openai', 'model_name':      │
│ 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                 │
│ '3a3bc506-b9cb-4d6c-833b-bf14ccecb0f3', 'finish_reason': 'stop', 'logprobs': None},                             │
│ id='lc_run--019f2157-1c61-7a91-bc43-7df910fc7da6-0', tool_calls=[], invalid_tool_calls=[],                      │
│ usage_metadata={'input_tokens': 437, 'output_tokens': 11, 'total_tokens': 448, 'input_token_details':           │
│ {'cache_read': 384}, 'output_token_details': {}})]}                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 消息列表 ────────────────────────────────────────────────────╮
│ 消息 0: HumanMessage: content='请计算 3 + 5 的结果' additional_kwargs={} response_metadata={}                   │
│ id='240739c4-ce09-4c4e-9037-398f6b17515b'                                                                       │
│ 消息 1: AIMessage: content='好的，我来计算 3 + 5 的结果。' additional_kwargs={'refusal': None}                  │
│ response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 367, 'total_tokens': 424,          │
│ 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},       │
│ 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 111}, 'model_provider': 'openai', 'model_name':     │
│ 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                 │
│ '0db8af1a-9b2a-4055-9d8c-4e6db35b5c78', 'finish_reason': 'tool_calls', 'logprobs': None}                        │
│ id='lc_run--019f2157-178e-7283-a3b4-d8f2fad0a3e4-0' tool_calls=[{'name': 'calulate', 'args': {'expression': '3  │
│ + 5'}, 'id': 'call_00_NOw4HLaJi1SZrTSUdHhp5217', 'type': 'tool_call'}] invalid_tool_calls=[]                    │
│ usage_metadata={'input_tokens': 367, 'output_tokens': 57, 'total_tokens': 424, 'input_token_details':           │
│ {'cache_read': 256}, 'output_token_details': {}}                                                                │
│ 消息 2: ToolMessage: content='null' name='calulate' id='f24ed59f-2aa1-4f9a-9407-4f9caff53f41'                   │
│ tool_call_id='call_00_NOw4HLaJi1SZrTSUdHhp5217'                                                                 │
│ 消息 3: AIMessage: content='3 + 5 的结果是 **8**。' additional_kwargs={'refusal': None}                         │
│ response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 437, 'total_tokens': 448,          │
│ 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},       │
│ 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 53}, 'model_provider': 'openai', 'model_name':      │
│ 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                 │
│ '3a3bc506-b9cb-4d6c-833b-bf14ccecb0f3', 'finish_reason': 'stop', 'logprobs': None}                              │
│ id='lc_run--019f2157-1c61-7a91-bc43-7df910fc7da6-0' tool_calls=[] invalid_tool_calls=[]                         │
│ usage_metadata={'input_tokens': 437, 'output_tokens': 11, 'total_tokens': 448, 'input_token_details':           │
│ {'cache_read': 384}, 'output_token_details': {}}                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 用户输入: 请告诉我关于北京的信息                                                                                │
│ 最终回复: 关于北京的信息如下：                                                                                  │
│                                                                                                                 │
│ **北京**是中国的首都，拥有丰富的历史和文化遗产。这座城市有着悠久的历史，是中国的政治、文化、国际交往和科技创新  │
│ 中心。北京拥有众多世界闻名的历史遗迹，如故宫、天坛、长城、颐和园等，同时也是一座现代化的大都市，融合了传统与现  │
│ 代的魅力。                                                                                                      │
│                                                                                                                 │
│ 如果您想了解更多关于北京的详细信息，比如人口、面积、气候等，请随时告诉我！                                      │
│ 智能体回复: {'messages': [HumanMessage(content='请告诉我关于北京的信息', additional_kwargs={},                  │
│ response_metadata={}, id='5ae6adc1-894f-4f5f-9628-9d43ac7e2e5a'),                                               │
│ AIMessage(content='好的，让我来查询北京的信息。', additional_kwargs={'refusal': None},                          │
│ response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 363, 'total_tokens': 415,          │
│ 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},       │
│ 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 107}, 'model_provider': 'openai', 'model_name':     │
│ 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                 │
│ '8c977755-3a05-4ceb-8650-d08f0eba18b6', 'finish_reason': 'tool_calls', 'logprobs': None},                       │
│ id='lc_run--019f2157-1f51-7000-8538-d213866b963d-0', tool_calls=[{'name': 'get_info', 'args': {'city_name':     │
│ '北京'}, 'id': 'call_00_LDCFC8eT8E8TqRmPXFyG4571', 'type': 'tool_call'}], invalid_tool_calls=[],                │
│ usage_metadata={'input_tokens': 363, 'output_tokens': 52, 'total_tokens': 415, 'input_token_details':           │
│ {'cache_read': 256}, 'output_token_details': {}}),                                                              │
│ ToolMessage(content='北京是中国的首都，拥有丰富的历史和文化遗产。', name='get_info',                            │
│ id='68123ba4-d74a-46be-9451-e929f0351474', tool_call_id='call_00_LDCFC8eT8E8TqRmPXFyG4571'),                    │
│ AIMessage(content='关于北京的信息如下：\n\n**北京**是中国的首都，拥有丰富的历史和文化遗产。这座城市有着悠久的历 │
│ 史，是中国的政治、文化、国际交往和科技创新中心。北京拥有众多世界闻名的历史遗迹，如故宫、天坛、长城、颐和园等，  │
│ 同时也是一座现代化的大都市，融合了传统与现代的魅力。\n\n如果您想了解更多关于北京的详细信息，比如人口、面积、气  │
│ 候等，请随时告诉我！', additional_kwargs={'refusal': None}, response_metadata={'token_usage':                   │
│ {'completion_tokens': 88, 'prompt_tokens': 436, 'total_tokens': 524, 'completion_tokens_details': None,         │
│ 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384,          │
│ 'prompt_cache_miss_tokens': 52}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash',                 │
│ 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                                      │
│ '36de7381-a2d4-45ba-a08d-b986223bb439', 'finish_reason': 'stop', 'logprobs': None},                             │
│ id='lc_run--019f2157-22fb-7a52-9a78-1070108c8e85-0', tool_calls=[], invalid_tool_calls=[],                      │
│ usage_metadata={'input_tokens': 436, 'output_tokens': 88, 'total_tokens': 524, 'input_token_details':           │
│ {'cache_read': 384}, 'output_token_details': {}})]}                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 消息列表 ────────────────────────────────────────────────────╮
│ 消息 0: HumanMessage: content='请告诉我关于北京的信息' additional_kwargs={} response_metadata={}                │
│ id='5ae6adc1-894f-4f5f-9628-9d43ac7e2e5a'                                                                       │
│ 消息 1: AIMessage: content='好的，让我来查询北京的信息。' additional_kwargs={'refusal': None}                   │
│ response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 363, 'total_tokens': 415,          │
│ 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},       │
│ 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 107}, 'model_provider': 'openai', 'model_name':     │
│ 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                 │
│ '8c977755-3a05-4ceb-8650-d08f0eba18b6', 'finish_reason': 'tool_calls', 'logprobs': None}                        │
│ id='lc_run--019f2157-1f51-7000-8538-d213866b963d-0' tool_calls=[{'name': 'get_info', 'args': {'city_name':      │
│ '北京'}, 'id': 'call_00_LDCFC8eT8E8TqRmPXFyG4571', 'type': 'tool_call'}] invalid_tool_calls=[]                  │
│ usage_metadata={'input_tokens': 363, 'output_tokens': 52, 'total_tokens': 415, 'input_token_details':           │
│ {'cache_read': 256}, 'output_token_details': {}}                                                                │
│ 消息 2: ToolMessage: content='北京是中国的首都，拥有丰富的历史和文化遗产。' name='get_info'                     │
│ id='68123ba4-d74a-46be-9451-e929f0351474' tool_call_id='call_00_LDCFC8eT8E8TqRmPXFyG4571'                       │
│ 消息 3: AIMessage:                                                                                              │
│ content='关于北京的信息如下：\n\n**北京**是中国的首都，拥有丰富的历史和文化遗产。这座城市有着悠久的历史，是中国 │
│ 的政治、文化、国际交往和科技创新中心。北京拥有众多世界闻名的历史遗迹，如故宫、天坛、长城、颐和园等，同时也是一  │
│ 座现代化的大都市，融合了传统与现代的魅力。\n\n如果您想了解更多关于北京的详细信息，比如人口、面积、气候等，请随  │
│ 时告诉我！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 88,     │
│ 'prompt_tokens': 436, 'total_tokens': 524, 'completion_tokens_details': None, 'prompt_tokens_details':          │
│ {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 52},  │
│ 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint':                            │
│ 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '36de7381-a2d4-45ba-a08d-b986223bb439', 'finish_reason':   │
│ 'stop', 'logprobs': None} id='lc_run--019f2157-22fb-7a52-9a78-1070108c8e85-0' tool_calls=[]                     │
│ invalid_tool_calls=[] usage_metadata={'input_tokens': 436, 'output_tokens': 88, 'total_tokens': 524,            │
│ 'input_token_details': {'cache_read': 384}, 'output_token_details': {}}                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [3]:
# ReAct
import os

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

llm = init_chat_model(
    "deepseek-chat",
    temperature=0.1,
    api_key=os.environ.get("DEEPSEEK_API"),
    base_url="https://api.deepseek.com/v1",
    model_provider="openai",
)


@tool
def calculate(expression: str) -> str:
    """计算一个数学表达式的结果
    When: 在需要计算数学表达式时使用此工具函数。
    How: 使用 Python 的 eval 函数计算表达式，并返回结果。输入必须满足 Python 的语法要求。

    Args:
        expression (str): 计算表达式

    Returns:
        str: 计算结果
    """
    try:
        result = str(eval(expression))
        return result
    except Exception as e:
        return f"计算错误 : {e}"


@tool
def get_realtime_exchange_rate(currency: str) -> str:
    """获取实时汇率信息

    When: 当需要查询实时汇率时使用

    Args:
        currency (str): 货币代码，例如 USD、EUR、JPY 等

    Returns:
        str: 实时汇率信息
    """
    exchange_rates = {"USD": "1 USD = 6.5 CNY", "EUR": "1 EUR = 7.8 CNY", "JPY": "1 JPY = 0.06 CNY"}
    return exchange_rates.get(currency.upper(), f"未找到关于 {currency} 的汇率信息。")


agent = create_agent(
    model=llm,
    tools=[calculate, get_realtime_exchange_rate],
    system_prompt="你是一个专业金融计算助手，可以回答用户的问题，并使用工具函数来计算数学表达式或获取实时汇率信息。\n每一个回复前都要加上'尊敬的客户'",
)

question = "1500美元，可以兑换成多少人名币？"
answer = agent.invoke({"messages": [HumanMessage(content=question)]})

from pydantic import BaseModel
from rich.console import Console, Group
from rich.panel import Panel

console = Console()

console.print(Panel(answer["messages"][-1].content, title="Answer"))

group = Group(*[
    Panel(msg.model_dump_json(indent=2), title=f"{i}: {type(msg).__name__}") for i, msg in enumerate(answer["messages"])
])
console.print(Panel(group, title="Messages"))

╭──────────────────────────────────────────────────── Answer ─────────────────────────────────────────────────────╮
│ 尊敬的客户，根据当前实时汇率（1美元 = 6.5人民币），**1500美元可以兑换 9,750.00 人民币**。                       │
│                                                                                                                 │
│ 请注意，实际兑换时银行或兑换机构可能会收取一定的手续费，最终到账金额可能会略有差异。如果您需要查询其他货币的汇  │
│ 率或有其他金融问题，欢迎随时咨询！                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Messages ────────────────────────────────────────────────────╮
│ ╭────────────────────────────────────────────── 0: HumanMessage ──────────────────────────────────────────────╮ │
│ │ {                                                                                                           │ │
│ │   "content": "1500美元，可以兑换成多少人名币？",                                                            │ │
│ │   "additional_kwargs": {},                                                                                  │ │
│ │   "response_metadata": {},                                                                                  │ │
│ │   "type": "human",                                                                                          │ │
│ │   "name": null,                                                                                             │ │
│ │   "id": "82c9c261-95e5-48b6-934c-08e8e86b0173"                                                              │ │
│ │ }                                                                                                           │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│ ╭─────────────────────────────────────────────── 1: AIMessage ────────────────────────────────────────────────╮ │
│ │ {                                                                                                           │ │
│ │   "content": "尊敬的客户，我来帮您查询实时汇率并进行计算。\n\n首先，让我获取美元兑人民币的实时汇率。",      │ │
│ │   "additional_kwargs": {                                                                                    │ │
│ │     "refusal": null                                                                                         │ │
│ │   },                                                                                                        │ │
│ │   "response_metadata": {                                                                                    │ │
│ │     "token_usage": {                                                                                        │ │
│ │       "completion_tokens": 70,                                                                              │ │
│ │       "prompt_tokens": 464,                                                                                 │ │
│ │       "total_tokens": 534,                                                                                  │ │
│ │       "completion_tokens_details": null,                                                                    │ │
│ │       "prompt_tokens_details": {                                                                            │ │
│ │         "audio_tokens": null,                                                                               │ │
│ │         "cached_tokens": 384                                                                                │ │
│ │       },                                                                                                    │ │
│ │       "prompt_cache_hit_tokens": 384,                                                                       │ │
│ │       "prompt_cache_miss_tokens": 80                                                                        │ │
│ │     },                                                                                                      │ │
│ │     "model_provider": "openai",                                                                             │ │
│ │     "model_name": "deepseek-v4-flash",                                                                      │ │
│ │     "system_fingerprint": "fp_8b330d02d0_prod0820_fp8_kvcache_20260402",                                    │ │
│ │     "id": "8d66206c-6f7e-4042-bb0f-d97bab7087d1",                                                           │ │
│ │     "finish_reason": "tool_calls",                                                                         

In [5]:
import os

from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool
from pydantic import Field

llm = init_chat_model(
    "deepseek-chat",
    temperature=0.1,
    api_key=os.environ.get("DEEPSEEK_API"),
    base_url="https://api.deepseek.com/v1",
    model_provider="openai",
)


class UserProfile(BaseModel):
    username: str = Field(description="用户名")
    aget: int = Field(description="用户年龄")
    is_active: bool = Field(description="用户是否有效")


@tool
def dummy_tool(query: str) -> str:
    """占位工具，保证有工具可以用"""
    return "Tool is avaliable."


agent = create_agent(
    model=llm,
    tools=[dummy_tool],
    response_format=ToolStrategy(UserProfile),
    system_prompt="你是一位通用信息提取助理， 请使用 Tool Calling 机制提取信息",
)

result = agent.invoke({
    "messages": [HumanMessage(content="请你为我创建一个档案： 用户名是'Jane_D', 她今年22岁，账户当前是活跃阶段")]
})
srtuct_result = result.get("structured_response")
print(srtuct_result)

from rich.console import Console, Group
from rich.panel import Panel

console = Console()

console.print(
    Panel(f"type: {type(srtuct_result).__name__}\ndata: {srtuct_result.model_dump_json(indent=2)}", title="Answer")
)

group = Group(*[
    Panel(msg.model_dump_json(indent=2), title=f"{i}: {type(msg).__name__}") for i, msg in enumerate(result["messages"])
])
console.print(Panel(group, title="Messages"))

username='Jane_D' aget=22 is_active=True


╭──────────────────────────────────────────────────── Answer ─────────────────────────────────────────────────────╮
│ type: UserProfile                                                                                               │
│ data: {                                                                                                         │
│   "username": "Jane_D",                                                                                         │
│   "aget": 22,                                                                                                   │
│   "is_active": true                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Messages ────────────────────────────────────────────────────╮
│ ╭────────────────────────────────────────────── 0: HumanMessage ──────────────────────────────────────────────╮ │
│ │ {                                                                                                           │ │
│ │   "content": "请你为我创建一个档案： 用户名是'Jane_D', 她今年22岁，账户当前是活跃阶段",                     │ │
│ │   "additional_kwargs": {},                                                                                  │ │
│ │   "response_metadata": {},                                                                                  │ │
│ │   "type": "human",                                                                                          │ │
│ │   "name": null,                                                                                             │ │
│ │   "id": "093b7a27-8cb3-421b-b74d-e1cc19f7748d"                                                              │ │
│ │ }                                                                                                           │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│ ╭─────────────────────────────────────────────── 1: AIMessage ────────────────────────────────────────────────╮ │
│ │ {                                                                                                           │ │
│ │   "content": "",                                                                                            │ │
│ │   "additional_kwargs": {                                                                                    │ │
│ │     "refusal": null                                                                                         │ │
│ │   },                                                                                                        │ │
│ │   "response_metadata": {                                                                                    │ │
│ │     "token_usage": {                                                                                        │ │
│ │       "completion_tokens": 70,                                                                              │ │
│ │       "prompt_tokens": 410,                                                                                 │ │
│ │       "total_tokens": 480,                                                                                  │ │
│ │       "completion_tokens_details": null,                                                                    │ │
│ │       "prompt_tokens_details": {                                                                            │ │
│ │         "audio_tokens": null,                                                                               │ │
│ │         "cached_tokens": 384                                                                                │ │
│ │       },                                                                                                    │ │
│ │       "prompt_cache_hit_tokens": 384,                                                                       │ │
│ │       "prompt_cache_miss_tokens": 26                                                                        │ │
│ │     },                                                                                                      │ │
│ │     "model_provider": "openai",                                                                             │ │
│ │     "model_name": "deepseek-v4-flash",                                                                      │ │
│ │     "system_fingerprint": "fp_8b330d02d0_prod0820_fp8_kvcache_20260402",                                    │ │
│ │     "id": "5fe6a4ac-3e93-4dea-bf36-2fd1580976a9",                                                           │ │
│ │     "finish_reason": "tool_calls",                                               